# Fintech Privacy Filter — Evaluation Notebook

Evaluates `msdakot/fintech-privacy-filter-v0` on the held-out test split of `msdakot/fintech-privacy-pii`.

**Output:** Per-label entity F1 and per-language F1, logged to W&B and printed as copy-paste README tables.

**Runtime:** ~20–40 min on T4. CPU also works (~2 hrs) — GPU not required for eval.

---
### Checklist before running
- [ ] Left sidebar → Key icon → Add secret `HF_TOKEN` (read access to `msdakot/fintech-privacy-pii`)
- [ ] Left sidebar → Key icon → Add secret `WANDB_API_KEY` (from wandb.ai/authorize)
- [ ] Run cells top to bottom

## Preparation

In [ ]:
import json
import os
import subprocess
from collections import defaultdict
from pathlib import Path

import wandb
from datasets import load_dataset
from google.colab import drive, userdata
from huggingface_hub import HfApi, snapshot_download

In [ ]:
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU detected — eval will run on CPU (slower, but works fine).')

In [ ]:
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/fintech-pii-eval-results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results dir: {RESULTS_DIR}')

In [ ]:
!pip install -q 'opf @ git+https://github.com/openai/privacy-filter.git' datasets huggingface_hub wandb
print('Dependencies installed.')

In [ ]:
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
user = HfApi(token=HF_TOKEN).whoami()
print(f"HF authenticated as: {user['name']}")

os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
wandb.login()
print('W&B authenticated.')

## Model & Data

In [ ]:
HF_MODEL_REPO = 'msdakot/fintech-privacy-filter-v0'
print(f'Downloading {HF_MODEL_REPO}...')
local_model = snapshot_download(repo_id=HF_MODEL_REPO, token=HF_TOKEN)
print(f'Model downloaded to: {local_model}')
print('Contents:', os.listdir(local_model))

In [ ]:
DATA_DIR = Path('/content/eval-data')
DATA_DIR.mkdir(exist_ok=True)

def record_to_opf(record):
    """Convert HF dataset record → opf JSONL format.
    HF:  spans = [{start, end, label}, ...]
    opf: spans = {label: [[start, end], ...], ...}
    """
    spans_dict = {}
    for span in record.get('spans', []):
        label = span['label']
        spans_dict.setdefault(label, []).append([span['start'], span['end']])
    return {
        'text': record['text'],
        'spans': spans_dict,
        'language': record.get('language', 'en'),
    }

print('Loading test split from msdakot/fintech-privacy-pii...')
ds = load_dataset('msdakot/fintech-privacy-pii', split='test', token=HF_TOKEN)
print(f'Test examples: {len(ds)}')

# Full test set
test_path = DATA_DIR / 'test.jsonl'
with open(test_path, 'w', encoding='utf-8') as f:
    for record in ds:
        f.write(json.dumps(record_to_opf(record), ensure_ascii=False) + '\n')
print(f'Full test set → {test_path}')

# Per-language subsets
lang_records = defaultdict(list)
for record in ds:
    lang = record.get('language', 'en')
    lang_records[lang].append(record)

lang_paths = {}
for lang, records in sorted(lang_records.items()):
    lang_path = DATA_DIR / f'test_{lang}.jsonl'
    with open(lang_path, 'w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record_to_opf(record), ensure_ascii=False) + '\n')
    lang_paths[lang] = str(lang_path)
    print(f'  {lang}: {len(records)} examples → {lang_path}')

## Evaluation

Runs `opf eval` on the full test set, then per-language subsets.

In [ ]:
# Confirm opf eval CLI flags before running
help_res = subprocess.run(['opf', 'eval', '--help'], capture_output=True, text=True)
print(help_res.stdout or help_res.stderr)

In [ ]:
def _get_overall_f1(results: dict) -> float:
    """Extract overall F1 — handles common opf eval output formats."""
    # {"overall": {"f1": 0.95}} or {"f1": 0.95}
    if 'overall' in results:
        return results['overall'].get('f1', 0.0)
    if 'f1' in results:
        return results['f1']
    # Walk one level
    for v in results.values():
        if isinstance(v, dict) and 'f1' in v:
            return v['f1']
    return 0.0


def _get_per_label_f1(results: dict) -> dict:
    """Extract per-label F1 dict from opf eval output."""
    for key in ('per_label', 'by_label', 'labels', 'entities'):
        if key in results and isinstance(results[key], dict):
            label_data = results[key]
            return {
                label: (v['f1'] if isinstance(v, dict) else v)
                for label, v in label_data.items()
            }
    return {}


def run_opf_eval(dataset_path: str, checkpoint: str, label: str) -> dict:
    """Run opf eval and return parsed results dict."""
    cmd = ['opf', 'eval', dataset_path, '--checkpoint', checkpoint]
    print(f'[{label}] Command: {" ".join(cmd)}')
    res = subprocess.run(cmd, capture_output=True, text=True)

    if res.returncode != 0:
        print(f'[{label}] STDERR:\n{res.stderr}')
        raise RuntimeError(f'opf eval failed (code {res.returncode}) on {label}')

    # Try stdout JSON first
    stdout = res.stdout.strip()
    if stdout:
        try:
            return json.loads(stdout)
        except json.JSONDecodeError:
            pass

    # Fall back to file opf eval may write alongside the checkpoint
    for fname in ('eval_results.json', 'eval_summary.json', 'results.json'):
        candidate = os.path.join(checkpoint, fname)
        if os.path.exists(candidate):
            with open(candidate) as f:
                return json.load(f)

    print(f'[{label}] Raw stdout:\n{stdout[:3000]}')
    raise RuntimeError(
        f'Could not parse opf eval output for {label}. '
        'Inspect stdout above and adjust parsing.'
    )


print('Helper functions defined.')

In [ ]:
print('Running full evaluation on test set...')
print('-' * 60)
full_results = run_opf_eval(str(test_path), local_model, 'full')
print('\nFull eval results:')
print(json.dumps(full_results, indent=2))

In [ ]:
LANGUAGES = ['en', 'es', 'sv', 'de', 'it', 'nl', 'fr']
lang_results = {}

for lang in LANGUAGES:
    if lang not in lang_paths:
        print(f'[{lang}] No test examples found — skipping.')
        continue
    lang_results[lang] = run_opf_eval(lang_paths[lang], local_model, lang)
    print(f'[{lang}] F1: {_get_overall_f1(lang_results[lang]):.4f}\n')

print('Per-language evaluation complete.')

## Log & Export Results

In [ ]:
overall_f1 = _get_overall_f1(full_results)
per_label_f1 = _get_per_label_f1(full_results)
lang_f1_vals = {lang: _get_overall_f1(res) for lang, res in lang_results.items()}

print(f'Overall entity F1: {overall_f1:.4f}')
print(f'\nPer-label F1 ({len(per_label_f1)} labels):')
for label, f1 in sorted(per_label_f1.items(), key=lambda x: -x[1]):
    print(f'  {label:40s} {f1:.4f}')

In [ ]:
run = wandb.init(
    project='fintech-privacy-filter',
    name='eval-fintech-v0',
    config={
        'model': 'msdakot/fintech-privacy-filter-v0',
        'dataset': 'msdakot/fintech-privacy-pii',
        'split': 'test',
        'languages': LANGUAGES,
    },
)

wandb.log({'overall_f1': overall_f1})

if per_label_f1:
    label_table = wandb.Table(
        columns=['label', 'f1'],
        data=sorted(per_label_f1.items())
    )
    wandb.log({'per_label_f1': label_table})

for lang, f1 in lang_f1_vals.items():
    wandb.log({f'f1_{lang}': f1})

if lang_f1_vals:
    lang_table = wandb.Table(
        columns=['language', 'f1'],
        data=sorted(lang_f1_vals.items())
    )
    wandb.log({'per_language_f1': lang_table})

print(f'Results logged. W&B run: {run.url}')

In [ ]:
eval_output = {
    'overall_f1': overall_f1,
    'per_label_f1': per_label_f1,
    'per_language_f1': lang_f1_vals,
    'raw': {'full': full_results, 'languages': lang_results},
}

results_path = os.path.join(RESULTS_DIR, 'eval_results.json')
with open(results_path, 'w') as f:
    json.dump(eval_output, f, indent=2)
print(f'Full results saved to {results_path}')

In [ ]:
FINTECH_LABELS = [
    'iban', 'bban', 'lei', 'isin', 'cusip',
    'swift_mt_ref', 'policy_number', 'vat_number', 'loan_number', 'crypto_address',
]
SHARED_FINANCIAL = [
    'account_number', 'bank_routing_number', 'credit_debit_card', 'swift_bic',
    'tax_id', 'ssn', 'national_id',
]
GENERAL_PII = [
    'first_name', 'last_name', 'email', 'phone_number', 'street_address',
    'date_of_birth', 'ipv4', 'url', 'user_name', 'password',
]

def print_f1_table(title, labels, f1_dict):
    print(f'**{title}:**')
    print('| Label | This model F1 |')
    print('|---|---|')
    for label in labels:
        f1 = f1_dict.get(label)
        val = f'{f1:.3f}' if f1 is not None else 'TBD'
        print(f'| `{label}` | {val} |')
    print()

print('### Entity-level F1 — copy into README\n')
print_f1_table('Fintech labels (new)', FINTECH_LABELS, per_label_f1)
print_f1_table('Shared financial labels', SHARED_FINANCIAL, per_label_f1)
print_f1_table('General PII', GENERAL_PII, per_label_f1)

print('### Per-language F1 — copy into README\n')
lang_display = {
    'en': 'English (en)', 'es': 'Spanish (es)', 'sv': 'Swedish (sv)',
    'de': 'German (de)', 'it': 'Italian (it)', 'nl': 'Dutch (nl)', 'fr': 'French (fr)',
}
print('| Language | This model F1 |')
print('|---|---|')
for lang in LANGUAGES:
    f1 = lang_f1_vals.get(lang)
    val = f'{f1:.3f}' if f1 is not None else 'n/a'
    print(f'| {lang_display.get(lang, lang)} | {val} |')

In [ ]:
wandb.finish()
print('W&B run complete. View at:', run.url)